# Mini-Project 2 — MediCare Outpatient Visits: End-to-End Data Analysis
**Python for Full Stack Data Science with AI & Generative AI · Naresh IT**
*Lead Trainer: Ajit Byru · 3 × 90-minute sessions after Module 15*

**The business problem.** MediCare's hospital administrator has one year of outpatient visit records (`clinic_visits_2025.csv`, ~2,400 visits) and four questions she cannot answer from the billing screen: *Where are patients waiting longest? Which departments actually earn what? Is the clinic getting busier? And what should we change in 2026?*

You are the analyst. **Deliverable:** this notebook, cleaned and complete; a one-paragraph recommendation; a 5-slide summary; a 2-minute pitch.

**Rules:** no `sklearn`, no models — insight, not prediction. Every cleaning decision is written down in the Data-Quality Log. Every question ends in a chart.

| Session | Focus | Sections |
|---|---|---|
| 1 | Load · profile · clean | 1–2 |
| 2 | Five guided questions | 3 |
| 3 | Your own question · recommendation · pitch | 4–6 |

---
## Session 1 — Section 1: Load and profile

Before cleaning anything, *look*. `shape`, `head`, `info`, `describe`, `isna`, `duplicated`, `value_counts` — the profiling checklist from Module 14. Write down every problem you find **before** fixing any of them.

### 1.1 Load
Read the CSV and print its shape and first rows.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

raw = pd.read_csv("clinic_visits_2026.csv")
print(raw.shape)
raw.head()

(2423, 13)


,visit_id,patient_id,visit_date,department,doctor,age,gender,city,wait_minutes,consult_minutes,fee,payment_mode,follow_up
0,V101246,P1873,24/11/2025,Pediatrics,Dr. Das,1,F,Warangal,28,5,560.0,UPI,No
1,V101350,P1692,2025-04-28,Dermatology,Dr. Menon,54,M,Hyderabad,9,29,850.0,Insurance,No
2,V102168,P1159,2025-01-31,General Medicine,Dr. Bose,53,M,Hyderabad,21,20,430.0,UPI,No
3,V101745,P1195,23-Jan-2025,Dermatology,Dr. Menon,79,M,Secunderabad,9,24,900.0,UPI,Yes
4,V101334,P1709,2025-08-25,Orthopedics,Dr. Nair,40,M,Warangal,16,20,940.0,Insurance,Yes


### 1.2 info() — which columns are the wrong type?
 **PREDICT:** visit_date should be a datetime. Will it be?

In [ ]:
# YOUR CODE HERE
raw.info()
#raw["visit_date"] = pd.to_datetime(raw["visit_date"])
#raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 2423 entries, 0 to 2422
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   visit_id         2423 non-null   str    
 1   patient_id       2423 non-null   str    
 2   visit_date       2423 non-null   str    
 3   department       2423 non-null   str    
 4   doctor           2423 non-null   str    
 5   age              2423 non-null   int64  
 6   gender           2423 non-null   str    
 7   city             2423 non-null   str    
 8   wait_minutes     2423 non-null   int64  
 9   consult_minutes  2423 non-null   int64  
 10  fee              2303 non-null   float64
 11  payment_mode     2423 non-null   str    
 12  follow_up        2423 non-null   str    
dtypes: float64(1), int64(3), str(9)
memory usage: 246.2 KB


### 1.3 describe() — anything impossible?
Look at min and max of every numeric column. A hospital dataset has physical limits.

In [ ]:
# YOUR CODE HERE
# Hint: raw.describe()
raw.describe()


,age,wait_minutes,consult_minutes,fee
count,2423.000000,2423.000000,2423.000000,2303.000000
mean,45.039208,28.637639,17.881965,1076.126791
std,24.484135,11.908123,6.252076,3697.973219
min,0.000000,-37.000000,5.000000,430.000000
25%,27.000000,21.000000,13.000000,560.000000
50%,45.000000,28.000000,18.000000,730.000000
75%,64.000000,37.000000,22.500000,1025.000000
max,412.000000,64.000000,35.000000,50000.000000


### 1.4 Missing values and duplicates

In [ ]:
# YOUR CODE HERE
# Hint: raw.isna().sum() and raw.duplicated().sum()
raw.isna().sum()
#raw.isnull().sum()
#raw.duplicated().sum()
#raw.isnull().sum().sum()
raw.duplicated().sum()



np.int64(23)

### 1.5 Category hygiene
value_counts() on department. How many departments does MediCare *actually* have?

In [ ]:
# YOUR CODE HERE
# Hint: raw["department"].value_counts()
raw["department"].value_counts()

department
General Medicine      784
Cardiology            375
Orthopedics           370
Dermatology           311
Pediatrics            302
ENT                   221
general medicine        8
 GENERAL MEDICINE       8
cardiology              8
 DERMATOLOGY            6
ent                     6
orthopedics             5
 ORTHOPEDICS            4
 ENT                    4
 CARDIOLOGY             4
 PEDIATRICS             4
dermatology             3
Name: count, dtype: int64

### 1.6 Peek at the dates
Sample a few raw `visit_date` strings. How many different formats can you see?

In [ ]:
# YOUR CODE HERE

raw["visit_date"].sample(10, random_state=1).tolist() ## use tolist convert data into list


['21/06/2025',
 '2025-04-02',
 '2025-11-11',
 '2025-11-06',
 '14/11/2025',
 '2025-12-25',
 '2025-10-02',
 '2025-08-14',
 '2025-04-14',
 '2025-08-13']

### 📝 Data-Quality Log (fill this in — it is graded)
Record every problem found in 1.1–1.6. Add the fix and reasoning as you clean in Section 2.

| # | Problem found | Rows affected | Fix applied | Why this fix (not another) |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |
| 4 | | | | |
| 5 | | | | |
| 6 | | | | |
| 7 | | | | |

---
## Section 2: Clean

Work on a **copy** (`df = raw.drop_duplicates().copy()`) so `raw` stays untouched for comparison. Order matters: duplicates first (so counts are honest), then text, dates, ranges, outliers, and finally imputation.

2.1 Duplicates
Drop exact duplicate rows. Print before / after / removed.

In [ ]:
# YOUR CODE HERE
df = raw.drop_duplicates().copy()
df

,visit_id,patient_id,visit_date,department,doctor,age,gender,city,wait_minutes,consult_minutes,fee,payment_mode,follow_up
0,V101246,P1873,24/11/2025,Pediatrics,Dr. Das,1,F,Warangal,28,5,560.0,UPI,No
1,V101350,P1692,2025-04-28,Dermatology,Dr. Menon,54,M,Hyderabad,9,29,850.0,Insurance,No
2,V102168,P1159,2025-01-31,General Medicine,Dr. Bose,53,M,Hyderabad,21,20,430.0,UPI,No
3,V101745,P1195,23-Jan-2025,Dermatology,Dr. Menon,79,M,Secunderabad,9,24,900.0,UPI,Yes
4,V101334,P1709,2025-08-25,Orthopedics,Dr. Nair,40,M,Warangal,16,20,940.0,Insurance,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2418,V101698,P1032,15/07/2025,Orthopedics,Dr. Nair,73,F,Secunderabad,5,20,910.0,UPI,No
2419,V100333,P1630,2025-05-26,Pediatrics,Dr. Reddy,12,F,Secunderabad,12,16,610.0,Insurance,No
2420,V101294,P1091,20/03/2025,General Medicine,Dr. Pillai,26,M,Secunderabad,19,16,430.0,Cash,No
2421,V101105,P1418,06-Oct-2025,Orthopedics,Dr. Nair,51,M,Hyderabad,32,19,960.0,UPI,No


 2.2 Department names
Strip whitespace and standardise case. Then check the result — did the standardisation break anything?

In [ ]:
# YOUR CODE HERE
# Hint: .str.strip().str.title()  -- then look at the unique values carefully
df["department"] = df["department"].str.strip().str.title()
df

,visit_id,patient_id,visit_date,department,doctor,age,gender,city,wait_minutes,consult_minutes,fee,payment_mode,follow_up
0,V101246,P1873,24/11/2025,Pediatrics,Dr. Das,1,F,Warangal,28,5,560.0,UPI,No
1,V101350,P1692,2025-04-28,Dermatology,Dr. Menon,54,M,Hyderabad,9,29,850.0,Insurance,No
2,V102168,P1159,2025-01-31,General Medicine,Dr. Bose,53,M,Hyderabad,21,20,430.0,UPI,No
3,V101745,P1195,23-Jan-2025,Dermatology,Dr. Menon,79,M,Secunderabad,9,24,900.0,UPI,Yes
4,V101334,P1709,2025-08-25,Orthopedics,Dr. Nair,40,M,Warangal,16,20,940.0,Insurance,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2418,V101698,P1032,15/07/2025,Orthopedics,Dr. Nair,73,F,Secunderabad,5,20,910.0,UPI,No
2419,V100333,P1630,2025-05-26,Pediatrics,Dr. Reddy,12,F,Secunderabad,12,16,610.0,Insurance,No
2420,V101294,P1091,20/03/2025,General Medicine,Dr. Pillai,26,M,Secunderabad,19,16,430.0,Cash,No
2421,V101105,P1418,06-Oct-2025,Orthopedics,Dr. Nair,51,M,Hyderabad,32,19,960.0,UPI,No


### 2.3 Dates — the trap
Three formats live in one column. The tempting one-liner is `pd.to_datetime(..., format="mixed", dayfirst=True)`. Run it and look at the weekday distribution.

🔮 **PREDICT:** MediCare is closed on Sundays. How many Sunday visits should there be?

In [ ]:
# [TRAP] The obvious one-liner. Looks fine...
parsed = pd.to_datetime(
    df["visit_date"],
    format="mixed",
    dayfirst=True
)
# date_time ...."01/01/2025" ---> 2025-01-01 the first is taken as date
##03/05/2026 ---> may 6, 2025 or june 5, 2025
## dayfirst = True 5 june 2025
## mixed ---- 5-6-2025or 5 mar 2025, 05/06/2025, june 5 2025
print(parsed.dtype)
print("Range", parsed.min().date(), "to", parsed.max().date())
print(parsed.dt.day_name().value_counts())

datetime64[us]
Range 2025-01-01 to 2025-12-31
visit_date
Monday       459
Friday       410
Tuesday      408
Wednesday    380
Thursday     355
Saturday     305
Sunday        83
Name: count, dtype: int64


### 2.4 Investigate
Compare the ISO-formatted strings with what the parser produced.

##dayFirt = True; 2025-11-12 it became 11-dec . only <= 12 get flipped

In [ ]:
# YOUR CODE HERE
# Hint: build a mask for ISO-looking strings with .str.match(r"\d{4}-\d{2}-\d{2}$"); compare parsed.dt.strftime("%Y-%m-%d") with the text

iso_rows = df["visit_date"].str.match(r"\d{4}-\d{2}-\d{2}$")
iso_text = df.loc[iso_rows,"visit_date"]
swapped = parsed[iso_rows].dt.strftime("%Y-%m-%d") != iso_text
print("ISO dates is silently day/months-swapped :", swapped.sum(), "of", iso_rows.sum())
print({"text": iso_text[swapped], "parsed":parsed[iso_rows][swapped].dt.date})

ISO dates is silently day/months-swapped : 596 of 1683
{'text': 6       2025-11-12
10      2025-02-12
12      2025-03-04
13      2025-04-09
14      2025-02-07
           ...    
2409    2025-09-06
2410    2025-11-06
2411    2025-08-09
2416    2025-08-07
2422    2025-09-12
Name: visit_date, Length: 596, dtype: str, 'parsed': 6       2025-12-11
10      2025-12-02
12      2025-04-03
13      2025-09-04
14      2025-07-02
           ...    
2409    2025-06-09
2410    2025-06-11
2411    2025-09-08
2416    2025-07-08
2422    2025-12-09
Name: visit_date, Length: 596, dtype: object}


 2.5 Fix — parse each format explicitly
Write a function that tries each format in turn with `errors="coerce"` and fills gaps. Verify: zero unparsed, zero Sundays.

In [ ]:
# YOUR CODE HERE
# Hint: pd.to_datetime(s, format="%Y-%m-%d", errors="coerce") then .fillna(...) with the next format
def iso_rows(s):
    parsed = pd.to_datetime(
        s,
        format="%Y-%m-%d",
        errors="coerce"
    )
    parsed = parsed.fillna(
        pd.to_datetime(
            s,
            format="%d/%m/%Y",
            errors="coerce"
        )
    )

    parsed = parsed.fillna(
        pd.to_datetime(
            s,
            format="%m/%d/%Y",
            errors="coerce"
        )
    )
    return parsed
print(parsed)

0      2025-11-24
1      2025-04-28
2      2025-01-31
3      2025-01-23
4      2025-08-25
          ...    
2418   2025-07-15
2419   2025-05-26
2420   2025-03-20
2421   2025-10-06
2422   2025-12-09
Name: visit_date, Length: 2400, dtype: datetime64[us]


2.6 Impossible ages
Find ages ≤ 0 or > 110. Decide: repair or reject? (There is no way to recover a true age — reject, and record the visit_ids.)

In [ ]:
# YOUR CODE HERE
# Hint: (df["age"] <= 0) | (df["age"] > 110)
iso_rows = df[(df["age"] <=0) | (df["age"] > 110)]
print(iso_rows[["visit_id", "department", "age"]])
df = df[(df["age"] > 0) & (df["age"] <= 110)]
print("rows offers age filters", len(df))

     visit_id        department  age
1629  V101228  General Medicine    0
2032  V102161        Cardiology  412
2251  V102242  General Medicine  199
rows offers age filters 2397


2.7 Negative wait times
A negative wait is a sign error at data entry, not a real value. Decide: repair (`abs`) or reject? Justify in the log.

In [ ]:
# YOUR CODE HERE
# Hint: df.loc[mask, "wait_minutes"] = ...abs()
negetive  = df["wait_minutes"] < 0
print("negetive wait rows", len(negetive))
df.loc[df["wait_minutes"] < 0, "wait_minutes"] = df.loc[df["wait_minutes"]<0,"wait_minutes"]
print("min wait now:", df["wait_minutes"].min())

negetive wait rows 2397
min wait now: 3


### 2.8 Fee outliers — the judgement call
Apply the 1.5 × IQR fence from Module 15. Then **look at what it flags** before doing anything.

In [ ]:
# YOUR CODE HERE
# Hint: df["fee"].quantile([0.25, 0.75]); fence = q3 + 1.5*iqr; look at department and unique fee values of flagged rows
Q1, Q3 = df["fee"].quantile([.25, 0.75])
IQR = Q3 - Q1
upper = Q3 + 1.5*IQR
flagged = df[df["fee"] > upper]
print(f"Q1->{Q1},Q3 -> {Q3}, IQR -> {IQR}, upper limit ={upper}")
print("rows above limit:", len(flagged))
print(flagged["department"].value_counts())

Q1->560.0,Q3 -> 1030.0, IQR -> 470.0, upper limit =1735.0
rows above limit: 13
department
Orthopedics    7
Cardiology     6
Name: count, dtype: int64


2.9 Decide
The fence flags them. Are they errors? Look at the values and the departments. Write your decision in the log.

In [ ]:
# YOUR CODE HERE
# Hint: df["is_procedure"] = ...

2.10 Missing fees
Impute with a median — but *which* median? 🔮 **PREDICT:** What goes wrong if you use the overall median for a Cardiology visit?

In [ ]:
# YOUR CODE HERE
# Hint: dept_median = df.groupby("department")["fee"].median(); df["department"].map(dept_median)


Missing fees: 0


 2.11 Derived columns and save
Add weekday and month. Save clinic_visits_clean.csv. Print the final shape — this is your acceptance number.                   

In [ ]:
# YOUR CODE HERE
# Make sure visit_date is a datetime column


C:\Users\thinkpad\AppData\Local\Temp\ipykernel_16116\2633628941.py:3: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["visit_date"] = pd.to_datetime(df["visit_date"])


ValueError: time data "2025-04-28" doesn't match format "%d/%m/%Y". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

---
## Session 2 — Section 3: Five guided questions

Each question ends in a chart with a title, axis labels and a one-sentence reading in the markdown cell below it. **A chart without a sentence is not an answer.**

### Q1. Which department is busiest, and on which weekday?
Two bar charts. Reading: what does Monday mean for staffing?

In [ ]:
# YOUR CODE HERE
# Hint: value_counts().plot(kind="bar"); reindex weekdays in order


*Reading:*

### Q2. Where do patients wait longest?
Group wait time by department: mean, median, std, and the 90th percentile. Box plot. Reading: which department's *worst* days matter most?

In [ ]:
# YOUR CODE HERE
# Hint: groupby("department")["wait_minutes"].agg(mean=..., median=..., std=..., p90=lambda s: s.quantile(0.9)); df.boxplot(column=..., by=...)


*Reading:*

### Q3. What does each department really earn per visit?
Mean vs median fee by department, with and without procedures. Reading: which number would you put in the annual report, and why?

In [ ]:
# YOUR CODE HERE
# Hint: agg(mean=..., median=..., max=...) with and without df["is_procedure"]; grouped bar via .plot(kind="bar")


*Reading:*

### Q4. Does patient age relate to consultation length?
Scatter, `r`, `np.polyfit` line. Reading: how much longer per decade of age?

In [ ]:
# YOUR CODE HERE
# Hint: np.corrcoef(...)[0, 1]; np.polyfit(..., deg=1); plt.scatter + plt.plot


*Reading:*

### Q5. Is the clinic getting busier? Monthly revenue trend
Line chart, all revenue vs consultations only. Reading: are the spikes growth or single events?

In [ ]:
# YOUR CODE HERE
# Hint: groupby("month")["fee"].sum() / 1e5, twice (all / non-procedure); plt.plot with marker


*Reading:*

---
## Session 3 — Section 4: Your own question

Pick **one** and take it further than the guided questions did — this is your headline finding and your slide 3.

1. **Doctor workload** — is any doctor carrying an unfair share of visits or wait time within their department?
2. **Follow-up compliance** — which departments/ages come back for follow-up, and which don't?
3. **Payment-mode trends** — is UPI displacing cash across the year? Does mode differ by city or age?
4. **City-wise access** — do out-of-city patients wait longer or pay more?
5. **Consultation efficiency** — wait vs consult ratio by department: where does the queue exceed the care?
6. **Revenue concentration** — what share of revenue comes from the top 10% of visits? Top 2 departments?

Requirements: at least one `groupby`, one chart, one statistic you can defend, and a sentence on what you'd need to be sure.

In [ ]:
# YOUR CODE HERE
# Hint: your question, your code


---
## Section 5: Recommendation

**One paragraph, 120–180 words, written for the administrator (not for your trainer).** Structure: the single most important finding → the number that proves it → what to change in 2026 → one thing you would check with more data.

> *Your paragraph here.*

---
## Section 6: Presentation

**Five slides** (template in `README.md`): 1 Question · 2 Data & cleaning · 3 Three findings · 4 Recommendation · 5 What I'd do next.
**Two-minute pitch** in class. **README** with the cleaning log summary and the five interview questions answered.

### The cliffhanger
`np.polyfit` fitted consult time from *one* column. The administrator asks: "Can you predict it from age **and** department **and** doctor together?" With what you know today — no. Module 16 — yes.